# 🏏 IPL Deliveries Data Analysis | Python Case Study

## Problem Statement
This project analyses IPL ball-by-ball delivery data to understand scoring patterns, extras, batting performance, bowling performance, wickets, and innings-level trends.

## Objectives
- Explore the deliveries dataset and its columns.
- Understand runs, extras, and wickets.
- Identify high-scoring batters and wicket-taking bowlers.
- Compare teams' batting and bowling performance.
- Visualise patterns using charts.
- Summarise findings from the dataset.

**Dataset:** `deliveries.csv` — each row represents a delivery/ball record.

## 1. Dataset Description

The deliveries dataset contains ball-by-ball information, including match ID, innings, batting team, bowling team, over, ball, batter, bowler, runs, extras, and dismissal information.

> Note: A delivery row is not the same as a match. To calculate innings totals correctly, group by both `match_id` and `inning`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

### Load the dataset

If `deliveries.csv` is already in the Colab runtime, the notebook will load it directly. Otherwise, it will ask you to upload the CSV file.

In [ ]:
file_name = "deliveries.csv"

if not os.path.exists(file_name):
    try:
        from google.colab import files
        uploaded = files.upload()
        file_name = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            "Please place deliveries.csv in the same folder as this notebook."
        )

deliveries = pd.read_csv(file_name)
print("Dataset loaded successfully.")

## 2. Dataset Exploration

In [ ]:
print("Dataset Shape:", deliveries.shape)
print("\nColumn Names:")
print(deliveries.columns.tolist())

In [ ]:
print("Dataset Information:")
deliveries.info()

In [ ]:
deliveries.head()

In [ ]:
deliveries.sample(min(5, len(deliveries)), random_state=42)

In [ ]:
print("Number of rows:", deliveries.shape[0])
print("Number of columns:", deliveries.shape[1])
print("Number of unique matches:", deliveries["match_id"].nunique())
print("Innings values:", sorted(deliveries["inning"].dropna().unique().tolist()))

## 3. Data Quality Check

The dataset is treated as already cleaned, as provided. These checks are included to document its current quality rather than make unnecessary changes.

In [ ]:
print("Missing values by column:")
display(deliveries.isnull().sum().to_frame("missing_values"))

In [ ]:
print("Duplicate rows:", deliveries.duplicated().sum())
print("\nData types:")
display(deliveries.dtypes.to_frame("data_type"))

## 4. Feature Classification

### Numerical features
Examples: `inning`, `over`, `ball`, `batsman_runs`, `extra_runs`, `total_runs`.

### Categorical/text features
Examples: `batting_team`, `bowling_team`, `batsman`, `bowler`, `dismissal_kind`.

The following cell classifies columns based on their current pandas data types.

In [ ]:
numeric_cols = deliveries.select_dtypes(include=np.number).columns.tolist()
categorical_cols = deliveries.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical columns:")
print(numeric_cols)
print("\nCategorical/Text columns:")
print(categorical_cols)

# 5. Exploratory Data Analysis (EDA)

## 5.1 Univariate Analysis — Numerical Features

Univariate analysis studies one variable at a time.

In [ ]:
numeric_analysis_cols = [
    col for col in [
        "inning", "over", "ball", "batsman_runs", "extra_runs",
        "total_runs", "wide_runs", "noball_runs", "bye_runs",
        "legbye_runs", "is_super_over"
    ] if col in deliveries.columns
]

display(deliveries[numeric_analysis_cols].describe().T)

In [ ]:
for col in ["batsman_runs", "extra_runs", "total_runs"]:
    if col in deliveries.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(deliveries[col], bins=20)
        plt.title(f"Distribution of {col.replace('_', ' ').title()}")
        plt.xlabel(col.replace("_", " ").title())
        plt.ylabel("Number of deliveries")
        plt.tight_layout()
        plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=deliveries["total_runs"])
plt.title("Boxplot of Total Runs per Delivery")
plt.xlabel("Total runs")
plt.tight_layout()
plt.show()

### Univariate Analysis — Categorical Features

We examine the most frequent teams, batters, bowlers, and dismissal types.

In [ ]:
categorical_analysis_cols = [
    col for col in [
        "batting_team", "bowling_team", "batsman", "bowler", "dismissal_kind"
    ] if col in deliveries.columns
]

for col in categorical_analysis_cols:
    print(f"\nTop values for {col}:")
    display(deliveries[col].value_counts(dropna=False).head(10).to_frame("count"))

In [ ]:
for col in ["batting_team", "bowling_team", "dismissal_kind"]:
    if col in deliveries.columns:
        counts = deliveries[col].value_counts().head(10)
        plt.figure(figsize=(10, 4))
        counts.plot(kind="bar")
        plt.title(f"Top 10 {col.replace('_', ' ').title()} Categories")
        plt.xlabel(col.replace("_", " ").title())
        plt.ylabel("Delivery count")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

## 5.2 Innings-Level Analysis

Each match may contain multiple innings. We create a match-innings summary so that runs and extras are not incorrectly combined across every match.

In [ ]:
innings_summary = (
    deliveries.groupby(["match_id", "inning"], as_index=False)
    .agg(
        total_runs=("total_runs", "sum"),
        batsman_runs=("batsman_runs", "sum"),
        extra_runs=("extra_runs", "sum"),
        deliveries=("ball", "count")
    )
)

innings_summary.head()

In [ ]:
print("Number of match-innings:", len(innings_summary))
display(innings_summary.groupby("inning")[["total_runs", "batsman_runs", "extra_runs"]].mean().round(2))

In [ ]:
# Average total runs by innings number
avg_runs_by_inning = innings_summary.groupby("inning")["total_runs"].mean().sort_index()

plt.figure(figsize=(8, 4))
plt.plot(avg_runs_by_inning.index, avg_runs_by_inning.values, marker="o")
plt.title("Average Total Runs by Innings")
plt.xlabel("Innings number")
plt.ylabel("Average total runs per match-innings")
plt.xticks(avg_runs_by_inning.index)
plt.tight_layout()
plt.show()

In [ ]:
# Total extras by innings number, summed across the dataset
extras_by_inning = deliveries.groupby("inning")["extra_runs"].sum().sort_index()

plt.figure(figsize=(8, 4))
plt.plot(extras_by_inning.index, extras_by_inning.values, marker="o")
plt.title("Total Extra Runs by Innings Number (All Matches Combined)")
plt.xlabel("Innings number")
plt.ylabel("Total extra runs")
plt.xticks(extras_by_inning.index)
plt.tight_layout()
plt.show()

## 5.3 Runs and Extras Analysis

In [ ]:
run_summary = deliveries[["batsman_runs", "extra_runs", "total_runs"]].sum().to_frame("total")
display(run_summary)

In [ ]:
extras_cols = [
    col for col in ["wide_runs", "noball_runs", "bye_runs", "legbye_runs", "penalty_runs"]
    if col in deliveries.columns
]
extras_summary = deliveries[extras_cols].sum().sort_values(ascending=False)
display(extras_summary.to_frame("runs"))

plt.figure(figsize=(8, 4))
extras_summary.plot(kind="bar")
plt.title("Total Runs by Extras Type")
plt.xlabel("Extras type")
plt.ylabel("Runs")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
print("Total extra runs:", int(deliveries["extra_runs"].sum()))
print("Total runs:", int(deliveries["total_runs"].sum()))
print("Percentage of runs from extras:",
      round(deliveries["extra_runs"].sum() / deliveries["total_runs"].sum() * 100, 2)
      if deliveries["total_runs"].sum() else 0, "%")

## 5.4 Batting Performance

In [ ]:
batter_runs = (
    deliveries.groupby("batsman")["batsman_runs"]
    .sum()
    .sort_values(ascending=False)
)

print("Top 10 run-scoring batters:")
display(batter_runs.head(10).to_frame("runs"))

In [ ]:
plt.figure(figsize=(10, 5))
batter_runs.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Batters by Total Runs")
plt.xlabel("Batsman runs")
plt.ylabel("Batter")
plt.tight_layout()
plt.show()

In [ ]:
# Number of fours and sixes hit by each batter
fours = deliveries[deliveries["batsman_runs"] == 4].groupby("batsman").size()
sixes = deliveries[deliveries["batsman_runs"] == 6].groupby("batsman").size()

boundary_summary = pd.DataFrame({
    "fours": fours,
    "sixes": sixes
}).fillna(0).astype(int)

boundary_summary["total_boundaries"] = boundary_summary["fours"] + boundary_summary["sixes"]
display(boundary_summary.sort_values("total_boundaries", ascending=False).head(10))

## 5.5 Bowling Performance

In [ ]:
# Wickets credited to the bowler; run-outs, retired hurt, obstructing the field,
# and similar dismissals are excluded from bowler wicket counts.
non_bowler_dismissals = [
    "run out", "retired hurt", "obstructing the field", "retired out"
]

wicket_deliveries = deliveries[
    deliveries["player_dismissed"].notna()
    & ~deliveries["dismissal_kind"].str.lower().isin(non_bowler_dismissals)
]

bowler_wickets = wicket_deliveries.groupby("bowler").size().sort_values(ascending=False)

print("Top 10 bowlers by wickets credited:")
display(bowler_wickets.head(10).to_frame("wickets"))

In [ ]:
plt.figure(figsize=(10, 5))
bowler_wickets.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Bowlers by Wickets Credited")
plt.xlabel("Wickets")
plt.ylabel("Bowler")
plt.tight_layout()
plt.show()

In [ ]:
print("Dismissal types:")
display(deliveries["dismissal_kind"].value_counts(dropna=True).to_frame("count"))

## 6. Bivariate Analysis

Bivariate analysis studies the relationship between two variables.

### 6.1 Numerical vs Numerical

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=deliveries, x="batsman_runs", y="total_runs", alpha=0.25)
plt.title("Batsman Runs vs Total Runs per Delivery")
plt.xlabel("Batsman runs")
plt.ylabel("Total runs")
plt.tight_layout()
plt.show()

In [ ]:
correlation_cols = [
    col for col in ["over", "ball", "batsman_runs", "extra_runs", "total_runs"]
    if col in deliveries.columns
]
plt.figure(figsize=(7, 5))
sns.heatmap(deliveries[correlation_cols].corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Selected Numerical Features")
plt.tight_layout()
plt.show()

### 6.2 Numerical vs Categorical

In [ ]:
team_runs = (
    deliveries.groupby("batting_team")["total_runs"]
    .sum()
    .sort_values(ascending=False)
)
display(team_runs.to_frame("total_runs"))

In [ ]:
plt.figure(figsize=(11, 5))
team_runs.sort_values().plot(kind="barh")
plt.title("Total Runs by Batting Team")
plt.xlabel("Total runs")
plt.ylabel("Batting team")
plt.tight_layout()
plt.show()

In [ ]:
team_avg_innings = (
    innings_summary.merge(
        deliveries[["match_id", "inning", "batting_team"]].drop_duplicates(),
        on=["match_id", "inning"],
        how="left"
    )
    .groupby("batting_team")["total_runs"]
    .mean()
    .sort_values(ascending=False)
)
display(team_avg_innings.to_frame("average_runs_per_innings"))

In [ ]:
plt.figure(figsize=(11, 5))
team_avg_innings.sort_values().plot(kind="barh")
plt.title("Average Runs per Match-Innings by Batting Team")
plt.xlabel("Average runs")
plt.ylabel("Batting team")
plt.tight_layout()
plt.show()

### 6.3 Team Matchups

In [ ]:
# Total runs scored by each batting team against each bowling team
team_matchup_runs = pd.pivot_table(
    deliveries,
    index="batting_team",
    columns="bowling_team",
    values="total_runs",
    aggfunc="sum",
    fill_value=0
)
display(team_matchup_runs)

## 7. Multivariate Analysis

Multivariate analysis considers three or more variables together.

In [ ]:
# Average runs by batting team and innings
team_inning_runs = (
    innings_summary.merge(
        deliveries[["match_id", "inning", "batting_team"]].drop_duplicates(),
        on=["match_id", "inning"],
        how="left"
    )
    .groupby(["batting_team", "inning"])["total_runs"]
    .mean()
    .unstack()
)
display(team_inning_runs.round(2))

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(team_inning_runs, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Average Runs by Batting Team and Innings")
plt.xlabel("Innings")
plt.ylabel("Batting team")
plt.tight_layout()
plt.show()

In [ ]:
# Wickets by innings and dismissal type
dismissal_data = deliveries[deliveries["dismissal_kind"].notna()]
dismissal_table = pd.crosstab(dismissal_data["inning"], dismissal_data["dismissal_kind"])
display(dismissal_table)

In [ ]:
plt.figure(figsize=(11, 5))
sns.heatmap(dismissal_table, annot=True, fmt="d", cmap="Oranges")
plt.title("Dismissal Types by Innings")
plt.xlabel("Dismissal type")
plt.ylabel("Innings")
plt.tight_layout()
plt.show()

## 8. Key Findings (Generated from the Dataset)

The following cells display the main results calculated from the uploaded data. Use these actual results when writing your final conclusions.

In [ ]:
print("PROJECT SUMMARY")
print("-" * 50)
print("Delivery records:", f"{len(deliveries):,}")
print("Unique matches:", f"{deliveries['match_id'].nunique():,}")
print("Total runs:", f"{int(deliveries['total_runs'].sum()):,}")
print("Total batsman runs:", f"{int(deliveries['batsman_runs'].sum()):,}")
print("Total extras:", f"{int(deliveries['extra_runs'].sum()):,}")

print("\nTop 5 run-scoring batters:")
display(batter_runs.head(5).to_frame("runs"))

print("\nTop 5 bowlers by wickets credited:")
display(bowler_wickets.head(5).to_frame("wickets"))

print("\nTop 5 batting teams by total runs:")
display(team_runs.head(5).to_frame("total_runs"))

print("\nMost common dismissal types:")
display(deliveries["dismissal_kind"].value_counts().head(5).to_frame("count"))

## 9. Conclusion

This project demonstrates how Python can be used to analyse IPL ball-by-ball data. The analysis covers scoring patterns, extras, batting and bowling performance, innings-level comparisons, dismissal types, and team matchups.

After running all cells, write a short conclusion based on the actual output. Avoid making claims that are not supported by the data.

## Tools Used
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Google Colab

## How to Run
1. Open this notebook in Google Colab.
2. Upload `deliveries.csv` when prompted (if it is not already in the runtime).
3. Select **Runtime → Run all**.
4. Review the tables, charts, and generated findings.